# [8.4] Circuit Tracing with Attribution Graphs

> **Notebooks: [exercises](./8.4_Circuit_Tracing_with_Attribution_Graphs_exercises.ipynb) | [solutions](./8.4_Circuit_Tracing_with_Attribution_Graphs_solutions.ipynb)**

In 8.2 you used EAP-style scores; in 8.3 you graded proposed circuits. Here you turn scores into a graph-shaped claim and make that claim falsifiable. By the end, you should be able to build a directed attribution graph, recover a source-to-target path, test target-metric explanation, compare a same-size alternative, and read the scoped CUDA result without overclaiming it.

<details>
<summary>Expected output</summary>

You should finish with a CPU graph contract containing a multi-hop toy path and a committed CUDA report whose selected edge is `position_5 -> position_5`.

</details>

<details>
<summary>Help - what is the core claim?</summary>

A graph claim is stronger than a score matrix but weaker than a full circuit proof. It says a small set of directed edges explains a metric, and it should survive perturbation, baseline, and counterfactual checks.

</details>

In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter8_automated_circuits"
section = "part4_circuit_tracing_attribution_graphs"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_circuit_tracing_attribution_graphs.tests as tests
import part4_circuit_tracing_attribution_graphs.utils as utils

GT_TIER = "GT-1"
EXERCISE_ID = "8_4_circuit_tracing_with_attribution_graphs"
EXPECTED_RUNTIME = "35-50 minutes for exercises; about 1-2 minutes for the CUDA preflight"
REQUIRES_GPU = True
DIFFICULTY = 4
IMPORTANCE = 4

CounterfactualDirection = Literal["increase", "decrease"]


@dataclass(frozen=True)
class CircuitTraceEdge:
    source: str
    target: str
    score: float


@dataclass(frozen=True)
class LocalAttributionGraph:
    nodes: tuple[str, ...]
    edges: tuple[CircuitTraceEdge, ...]


@dataclass(frozen=True)
class GraphMetricReport:
    full_metric: float
    corrupt_metric: float
    graph_metric: float
    explained_fraction: float
    explains_target_metric: bool


@dataclass(frozen=True)
class PathPerturbationReport:
    original_metric: float
    perturbed_metric: float
    metric_drop: float
    top_path_survives_test: bool


@dataclass(frozen=True)
class AlternativeGraphBaselineReport:
    graph_metric: float
    alternative_metric: float
    margin: float
    alternative_baseline_fails: bool


@dataclass(frozen=True)
class GraphSummaryCounterfactualReport:
    predicted_direction: CounterfactualDirection
    observed_delta: float
    predicts_counterfactual: bool


@dataclass(frozen=True)
class AttributionPathReport:
    source: str
    target: str
    path: tuple[str, ...]
    edge_scores: tuple[float, ...]
    path_score: float
    reaches_target: bool


## Validation Loop

The local contract is deliberately small:

```text
clean/corrupt cache + corrupt gradient -> EAP edge matrix -> top-k graph
                                         -> metric, path, baseline, counterfactual checks
```

<details>
<summary>Limitations</summary>

The committed CUDA path is a residual-position graph preflight. It is not sparse-feature circuit tracing, transcoder graph tracing, IOI path-patching replication, or broad OOD validation.

</details>

## Exercise 1 - EAP-style Edge Scores

Implement a directed upstream-by-downstream edge-score matrix.

<details>
<summary>Expected output</summary>

```text
All tests in `test_edge_attribution_scores_forms_position_edge_matrix` passed!
All tests in `test_edge_attribution_scores_rejects_degenerate_inputs` passed!
```

</details>

<details>
<summary>Help - why directed?</summary>

Rows are upstream activation deltas; columns are downstream gradients. Reversing them changes the graph claim.

</details>

<details>
<summary>Common bug</summary>

The common mistake is to return an elementwise product or a single scalar instead of a source-by-target matrix.

</details>

<details>
<summary>Solution</summary>

```python
if upstream_activation_delta.ndim != 2 or downstream_gradients.ndim != 2:
    raise ValueError("inputs must have shape (components, d_model).")
if upstream_activation_delta.numel() == 0 or downstream_gradients.numel() == 0:
    raise ValueError("inputs must be non-empty.")
if not t.isfinite(upstream_activation_delta).all() or not t.isfinite(downstream_gradients).all():
    raise ValueError("inputs must be finite.")
if upstream_activation_delta.shape[-1] != downstream_gradients.shape[-1]:
    raise ValueError("upstream and downstream hidden dimensions must match.")
return upstream_activation_delta.float() @ downstream_gradients.float().T
```

</details>

In [ ]:
def edge_attribution_scores(
    upstream_activation_delta: t.Tensor,
    downstream_gradients: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_edge_attribution_scores_forms_position_edge_matrix(edge_attribution_scores)
tests.test_edge_attribution_scores_rejects_degenerate_inputs(edge_attribution_scores)

## Exercise 2 - Build a Local Attribution Graph

Keep the largest directed nonzero edges while preserving node names.

<details>
<summary>Expected output</summary>

```text
All tests in `test_build_local_attribution_graph_keeps_top_directed_edges` passed!
All tests in `test_build_local_attribution_graph_rejects_bad_nodes_or_scores` passed!
```

</details>

<details>
<summary>Help - what does top-k prove?</summary>

Nothing by itself. It only proposes a compact graph to test with metrics and interventions.

</details>

<details>
<summary>Common bug</summary>

Flattening the matrix and forgetting how to recover source and target indices.

</details>

<details>
<summary>Solution</summary>

```python
flat_scores = edge_scores.flatten().float()
top_values, top_indices = flat_scores.topk(k=min(top_k, flat_scores.numel()))
num_nodes = edge_scores.shape[0]
# source = flat_index // num_nodes, target = flat_index % num_nodes
```

</details>

In [ ]:
def build_local_attribution_graph(
    edge_scores: t.Tensor,
    node_names: list[str],
    *,
    top_k: int = 3,
) -> LocalAttributionGraph:
    raise NotImplementedError()


tests.test_build_local_attribution_graph_keeps_top_directed_edges(build_local_attribution_graph)
tests.test_build_local_attribution_graph_rejects_bad_nodes_or_scores(build_local_attribution_graph)

## Exercise 3 - Recover the Top Directed Path

A graph claim often names a path, not just isolated edges. Find the highest-scoring directed source-to-target path.

<details>
<summary>Expected output</summary>

```text
All tests in `test_top_attribution_path_recovers_multi_hop_chain` passed!
All tests in `test_top_attribution_path_rejects_invalid_graphs` passed!
```

</details>

<details>
<summary>Help - why multiply edge strengths?</summary>

A path should be penalized for weak links. A sum can hide weak links behind one large edge; a product cannot.

</details>

<details>
<summary>Common bug</summary>

Searching an undirected graph, or returning the best edge even when it does not reach the target.

</details>

<details>
<summary>Solution</summary>

Use depth-first search, keep a visited path to avoid cycles, and update the best path only when `current == target` and at least one edge has been used.

</details>

In [ ]:
def top_attribution_path(
    graph: LocalAttributionGraph,
    *,
    source: str,
    target: str,
    max_depth: int = 4,
) -> AttributionPathReport:
    raise NotImplementedError()


tests.test_top_attribution_path_recovers_multi_hop_chain(top_attribution_path)
tests.test_top_attribution_path_rejects_invalid_graphs(top_attribution_path)

## Exercise 4 - Target Metric Explanation

Normalize the graph metric by the clean-corrupt gap.

<details>
<summary>Expected output</summary>

```text
All tests in `test_graph_metric_report_measures_explained_fraction` passed!
All tests in `test_metric_reports_reject_nonfinite_or_negative_thresholds` passed!
```

</details>

<details>
<summary>Help - what is the denominator?</summary>

The denominator is the behavior missing from the corrupt run: `full_metric - corrupt_metric`.

</details>

<details>
<summary>Common bug</summary>

Dividing by `full_metric` instead of the clean-corrupt gap.

</details>

<details>
<summary>Solution</summary>

```python
denominator = full_metric - corrupt_metric
if denominator == 0:
    raise ValueError("full_metric and corrupt_metric must differ.")
explained_fraction = (graph_metric - corrupt_metric) / denominator
```

</details>

In [ ]:
def graph_metric_report(
    *,
    full_metric: float,
    corrupt_metric: float,
    graph_metric: float,
    min_explained_fraction: float = 0.75,
) -> GraphMetricReport:
    raise NotImplementedError()


tests.test_graph_metric_report_measures_explained_fraction(graph_metric_report)
tests.test_metric_reports_reject_nonfinite_or_negative_thresholds(graph_metric_report)

## Exercise 5 - Perturbation and Alternative Baselines

A graph path should matter causally, and a same-size alternative should not explain the metric just as well.

<details>
<summary>Expected output</summary>

```text
All tests in `test_path_perturbation_and_alternative_baseline_reports` passed!
```

</details>

<details>
<summary>Help - why use a same-size alternative?</summary>

It rules out an easy failure mode: the metric might be so forgiving that any same-size graph works.

</details>

<details>
<summary>Common bug</summary>

Reversing the sign of `metric_drop`.

</details>

<details>
<summary>Solution</summary>

```python
metric_drop = original_metric - perturbed_metric
margin = graph_metric - alternative_metric
```

</details>

In [ ]:
def path_perturbation_report(
    *,
    original_metric: float,
    perturbed_metric: float,
    min_metric_drop: float = 0.5,
) -> PathPerturbationReport:
    raise NotImplementedError()


def alternative_graph_baseline_report(
    *,
    graph_metric: float,
    alternative_metric: float,
    min_margin: float = 0.5,
) -> AlternativeGraphBaselineReport:
    raise NotImplementedError()


tests.test_path_perturbation_and_alternative_baseline_reports(
    path_perturbation_report,
    alternative_graph_baseline_report,
)
tests.test_metric_reports_reject_nonfinite_or_negative_thresholds(
    path_perturbation_report=path_perturbation_report,
    alternative_graph_baseline_report=alternative_graph_baseline_report,
)

## Exercise 6 - Counterfactual Summaries

A written graph summary should predict the direction of an intervention before you see it.

<details>
<summary>Expected output</summary>

```text
All tests in `test_counterfactual_summary_report_checks_direction` passed!
```

</details>

<details>
<summary>Help - why test prose?</summary>

The last step of interpretability is often a written story. If the story is real, it should make a prediction.

</details>

<details>
<summary>Common bug</summary>

Treating a zero delta as both an increase and a decrease.

</details>

<details>
<summary>Solution</summary>

```python
observed_delta = counterfactual_metric - baseline_metric
predicts_counterfactual = observed_delta < 0  # for predicted_direction == "decrease"
```

</details>

In [ ]:
def graph_summary_counterfactual_report(
    *,
    predicted_direction: CounterfactualDirection,
    baseline_metric: float,
    counterfactual_metric: float,
) -> GraphSummaryCounterfactualReport:
    raise NotImplementedError()


tests.test_counterfactual_summary_report_checks_direction(graph_summary_counterfactual_report)

## Notebook Contract and Signature Result

The CPU smoke test should return JSON-serializable graph, metric, path, alternative, and counterfactual reports. The committed CUDA report then checks the same shape on a pinned `gelu-1l` preflight.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
edge: position_5 -> position_5
explained_fraction: 1.0
path_drop: 6.2771
alternative_margin: 6.2771
counterfactual_delta: -6.2771
```

</details>

<details>
<summary>Help - interpreting the Signature Result</summary>

The signature result proves the graph-tracing harness is wired to a real CUDA model path. It does not prove a rich multi-hop circuit, because the accepted graph has one residual-position edge.

</details>

In [ ]:
def graph_smoke_test() -> dict:
    edge_scores = t.tensor([[0.0, 0.8, 0.1], [0.0, 0.0, 0.9], [0.0, 0.0, 0.0]])
    node_names = ["feature:fact", "transcoder:mlp", "logit:Paris"]
    graph = build_local_attribution_graph(edge_scores, node_names, top_k=2)
    return {
        "nodes": list(graph.nodes),
        "edges": [(edge.source, edge.target, round(edge.score, 6)) for edge in graph.edges],
    }


def graph_metric_smoke_test() -> dict:
    return graph_metric_report(full_metric=3.0, corrupt_metric=0.0, graph_metric=2.4).__dict__


def path_perturbation_smoke_test() -> dict:
    return path_perturbation_report(original_metric=2.4, perturbed_metric=0.7, min_metric_drop=1.0).__dict__


def alternative_graph_smoke_test() -> dict:
    return alternative_graph_baseline_report(graph_metric=2.4, alternative_metric=1.0, min_margin=1.0).__dict__


def counterfactual_smoke_test() -> dict:
    return graph_summary_counterfactual_report(predicted_direction="decrease", baseline_metric=2.4, counterfactual_metric=0.8).__dict__


def path_smoke_test() -> dict:
    edge_scores = t.tensor([
        [0.0, 0.8, 0.2, 0.0],
        [0.0, 0.0, 0.9, 0.1],
        [0.0, 0.0, 0.0, 0.7],
        [0.0, 0.0, 0.0, 0.0],
    ])
    node_names = ["feature:subject", "transcoder:mlp", "feature:object", "logit:Paris"]
    graph = build_local_attribution_graph(edge_scores, node_names, top_k=5)
    return top_attribution_path(graph, source="feature:subject", target="logit:Paris", max_depth=3).__dict__


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "graph": graph_smoke_test(),
        "metric": graph_metric_smoke_test(),
        "path_perturbation": path_perturbation_smoke_test(),
        "alternative": alternative_graph_smoke_test(),
        "counterfactual": counterfactual_smoke_test(),
        "path": path_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)

In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    assert report["accepted"] and report["tests_passed"]
    assert gpu["preflight_passed"]
    assert gpu["graph_edge_source"] == "position_5"
    assert gpu["graph_edge_target"] == "position_5"
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
utils.print_report(
    "Committed CUDA circuit-tracing report",
    {
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "edge": f'{gpu["graph_edge_source"]} -> {gpu["graph_edge_target"]}',
        "edge_score": round(gpu["graph_edge_score"], 4),
        "explained_fraction": gpu["explained_fraction"],
        "path_drop": round(gpu["path_metric_drop"], 4),
        "alternative_margin": round(gpu["alternative_baseline_margin"], 4),
        "counterfactual_delta": round(gpu["counterfactual_observed_delta"], 4),
        "peak_vram_gb": round(gpu["peak_vram_gb"], 4),
    },
)

## Limitations and Further Research

<details>
<summary>Limitations</summary>

This notebook validates graph-tracing mechanics on toy contracts and one residual-position CUDA preflight. It does not claim full sparse-feature/transcoder circuit tracing, broad OOD robustness, or IOI-scale circuit discovery.

</details>

<details>
<summary>Further Research</summary>

Replace residual-position nodes with attention heads, SAE features, or transcoder features; sweep graph size; pre-register counterfactual predictions; and compare against known IOI path-patching evidence.

</details>